In [36]:
from river.drift import page_hinkley
import numpy as np

In [37]:
ph = page_hinkley.PageHinkley(min_instances=30, delta=0.01, threshold=25.0)

In [38]:
seed = 42
np.random.seed(seed)

for i in range(2500):
    # Simulating Z
    u_zy = np.random.binomial(1, 0.61)
    uz = np.random.binomial(1, 0.88)
    obs = u_zy ^ uz
    ph.update(obs)
    if ph.drift_detected:
        print(f"Drift detected at step {i}")
seed += 1
np.random.seed(seed)

In [39]:
cpds = {}

In [40]:
p_s = 0.9
p_t = 0.85
p_w = 0.4
p_x = 0.9
p_z =0.88
p_y =  0.9
p_wx=0.9
p_zy = 0.61

In [41]:
mean_t1z1 = 0
mean_t1z0 = 0
mean_t0z1 = 0
mean_t0z0 = 0
steps = 10000
for i in range(steps):
    # Simulating X
    u_x = np.random.binomial(1, 0.9)
    u_wx = np.random.binomial(1, 0.1)

    t = np.random.binomial(1, 0.85)

    uz = np.random.binomial(1, 0.88)
    u_zy = np.random.binomial(1, 0.61)
    z = u_zy ^ uz

    cfg = (z,t)
    if cfg not in cpds:
        cpds[cfg] = page_hinkley.PageHinkley(min_instances=30, delta=0.01, threshold=25.0)

    cpd = cpds[cfg]
    obs = u_wx & (u_x | t) ^ z
    match cfg:
        case (1, 1):
            mean_t1z1 += obs
        case (0, 1):
            mean_t1z0 += obs
        case (1, 0):
            mean_t0z1 += obs
        case (0, 0):
            mean_t0z0 += obs
    cpd.update(obs)
    if cpd.drift_detected:
        print(f"Drift detected at step {i}")

In [42]:
print("Means;")
print(f"Mean for (1, 1): {mean_t1z1 / steps}")
print(f"Mean for (0, 1): {mean_t1z0 / steps}")
print(f"Mean for (1, 0): {mean_t0z1 / steps}")
print(f"Mean for (0, 0): {mean_t0z0 / steps}")

Means;
Mean for (1, 1): 0.3172
Mean for (0, 1): 0.0502
Mean for (1, 0): 0.061
Mean for (0, 0): 0.0085


Means;
Mean for (1, 1): 0.0373
Mean for (0, 1): 0.4409
Mean for (1, 0): 0.0133
Mean for (0, 0): 0.0694